# Notebook 3 – Adult Hypertension Data

## Objective

Download, clean, quality check, and export age-standardized adult hypertension prevalence data for Pacific Island jurisdictions.

### Source

NCD Risk Factor Collaboration (NCD-RisC)

### Study Population

16 Pacific Island jurisdictions

### Output

processed_data/adult_hypertension.csv

## 1. Initialize project and download source data

In [1]:
import pandas as pd
import requests
from io import StringIO
import os

os.makedirs("raw_data/NCD_RisC", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

url = "https://www.ncdrisc.org/downloads/hypertension/NCD-RisC_Lancet_2021_Hypertension_age_standardised_countries.csv"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=60)

print("Status code:", response.status_code)
print("Downloaded MB:", round(len(response.content) / 1_000_000, 2))

response.raise_for_status()

hypertension_raw = pd.read_csv(StringIO(response.text))

hypertension_raw.to_csv(
    "raw_data/NCD_RisC/ncdrisc_hypertension_age_standardised_countries_raw.csv",
    index=False
)

print("Rows downloaded:", len(hypertension_raw))
print("Columns:")
print(hypertension_raw.columns.tolist())

hypertension_raw.head()

Status code: 200
Downloaded MB: 4.02
Rows downloaded: 12000
Columns:
['Country/Region/World', 'ISO', 'Sex', 'Year', 'Age', 'Prevalence of hypertension', 'Prevalence of hypertension lower 95% uncertainty interval', 'Prevalence of hypertension upper 95% uncertainty interval', 'Proportion of diagnosed hypertension among all hypertension', 'Proportion of diagnosed hypertension among all hypertension lower 95% uncertainty interval', 'Proportion of diagnosed hypertension among all hypertension upper 95% uncertainty interval', 'Proportion of treated hypertension among all hypertension', 'Proportion of treated hypertension among all hypertension lower 95% uncertainty interval', 'Proportion of treated hypertension among all hypertension upper 95% uncertainty interval', 'Proportion of controlled hypertension among all hypertension', 'Proportion of controlled hypertension among all hypertension lower 95% uncertainty interval', 'Proportion of controlled hypertension among all hypertension upper 95

,Country/Region/World,ISO,Sex,Year,Age,Prevalence of hypertension,Prevalence of hypertension lower 95% uncertainty interval,Prevalence of hypertension upper 95% uncertainty interval,Proportion of diagnosed hypertension among all hypertension,Proportion of diagnosed hypertension among all hypertension lower 95% uncertainty interval,Proportion of diagnosed hypertension among all hypertension upper 95% uncertainty interval,Proportion of treated hypertension among all hypertension,Proportion of treated hypertension among all hypertension lower 95% uncertainty interval,Proportion of treated hypertension among all hypertension upper 95% uncertainty interval,Proportion of controlled hypertension among all hypertension,Proportion of controlled hypertension among all hypertension lower 95% uncertainty interval,Proportion of controlled hypertension among all hypertension upper 95% uncertainty interval,Proportion of untreated stage 2 hypertension among all hypertension,Proportion of untreated stage 2 hypertension among all hypertension lower 95% uncertainty interval,Proportion of untreated stage 2 hypertension among all hypertension upper 95% uncertainty interval
0,Afghanistan,AFG,Men,1990,Age standardised (30-79 years),0.331843,0.190057,0.495322,0.217042,0.080527,0.416364,0.137442,0.030317,0.316249,0.026010,0.001348,0.105405,0.325092,0.169098,0.500993
1,Afghanistan,AFG,Men,1991,Age standardised (30-79 years),0.332705,0.196482,0.487221,0.222470,0.086410,0.414061,0.142342,0.034584,0.318273,0.026776,0.001727,0.102808,0.319920,0.171497,0.483893
2,Afghanistan,AFG,Men,1992,Age standardised (30-79 years),0.333643,0.202168,0.481609,0.228117,0.092876,0.415406,0.147571,0.039084,0.319533,0.027751,0.002184,0.101488,0.314806,0.172962,0.470403
3,Afghanistan,AFG,Men,1993,Age standardised (30-79 years),0.334619,0.209102,0.476224,0.233977,0.099647,0.416563,0.153131,0.043954,0.320049,0.028945,0.002624,0.099899,0.309792,0.173026,0.457284
4,Afghanistan,AFG,Men,1994,Age standardised (30-79 years),0.335512,0.214306,0.472759,0.239990,0.107187,0.418237,0.159044,0.049133,0.324701,0.030371,0.003202,0.100960,0.304867,0.174121,0.445977


## 2. Filter to study population

In [2]:
pacific_iso3 = [
    "ASM", "COK", "FJI", "FSM", "PYF",
    "KIR", "MHL", "NRU", "NIU",
    "PLW", "WSM", "SLB", "TKL",
    "TON", "TUV", "VUT"
]

hypertension_pacific = hypertension_raw[
    hypertension_raw["ISO"].isin(pacific_iso3)
].copy()

hypertension_pacific = hypertension_pacific.rename(columns={
    "Country/Region/World": "Country",
    "ISO": "ISO3",
    "Prevalence of hypertension": "Hypertension_Pct",
    "Proportion of diagnosed hypertension among all hypertension": "Hypertension_Diagnosed_Pct",
    "Proportion of treated hypertension among all hypertension": "Hypertension_Treated_Pct",
    "Proportion of controlled hypertension among all hypertension": "Hypertension_Controlled_Pct",
    "Proportion of untreated stage 2 hypertension among all hypertension": "Untreated_Stage2_Pct"
})

# Average male and female estimates to create one country-year value.
hypertension_pacific = (
    hypertension_pacific
    .groupby(["ISO3", "Country", "Year"], as_index=False)
    .agg({
        "Hypertension_Pct": "mean",
        "Hypertension_Diagnosed_Pct": "mean",
        "Hypertension_Treated_Pct": "mean",
        "Hypertension_Controlled_Pct": "mean",
        "Untreated_Stage2_Pct": "mean"
    })
)

print("Rows:", len(hypertension_pacific))
hypertension_pacific.head(20)

Rows: 480


,ISO3,Country,Year,Hypertension_Pct,Hypertension_Diagnosed_Pct,Hypertension_Treated_Pct,Hypertension_Controlled_Pct,Untreated_Stage2_Pct
0,ASM,American Samoa,1990,0.451845,0.349203,0.278335,0.054739,0.282912
1,ASM,American Samoa,1991,0.448751,0.350374,0.277502,0.054448,0.278761
2,ASM,American Samoa,1992,0.445737,0.351633,0.276891,0.054507,0.274769
3,ASM,American Samoa,1993,0.442786,0.352968,0.276507,0.054881,0.270942
4,ASM,American Samoa,1994,0.439933,0.354604,0.276286,0.055612,0.267244
5,ASM,American Samoa,1995,0.437227,0.356248,0.276227,0.056648,0.263688
6,ASM,American Samoa,1996,0.434858,0.357884,0.276255,0.057904,0.260279
7,ASM,American Samoa,1997,0.432879,0.359747,0.276480,0.059354,0.257074
8,ASM,American Samoa,1998,0.431367,0.361757,0.276915,0.060945,0.254119
9,ASM,American Samoa,1999,0.430388,0.364056,0.277519,0.062613,0.251483


## 3. Quality assurance

In [3]:
print(hypertension_pacific.info())

print()

print("Countries:")
print(sorted(hypertension_pacific["Country"].unique()))

print()

print("Years:")
print(hypertension_pacific["Year"].min(), "to", hypertension_pacific["Year"].max())

print()

print("Missing values by column:")
print(hypertension_pacific.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ISO3                         480 non-null    object 
 1   Country                      480 non-null    object 
 2   Year                         480 non-null    int64  
 3   Hypertension_Pct             480 non-null    float64
 4   Hypertension_Diagnosed_Pct   480 non-null    float64
 5   Hypertension_Treated_Pct     480 non-null    float64
 6   Hypertension_Controlled_Pct  480 non-null    float64
 7   Untreated_Stage2_Pct         480 non-null    float64
dtypes: float64(5), int64(1), object(2)
memory usage: 30.1+ KB
None

Countries:
['American Samoa', 'Cook Islands', 'Fiji', 'French Polynesia', 'Kiribati', 'Marshall Islands', 'Micronesia (Federated States of)', 'Nauru', 'Niue', 'Palau', 'Samoa', 'Solomon Islands', 'Tokelau', 'Tonga', 'Tuvalu', 'Vanuatu']

Years:
19

## 4. Coverage assessment

In [4]:
coverage = (
    hypertension_pacific
    .groupby(["ISO3", "Country"])
    .agg(
        First_Year=("Year", "min"),
        Last_Year=("Year", "max"),
        Years=("Year", "count")
    )
    .reset_index()
)

coverage

,ISO3,Country,First_Year,Last_Year,Years
0,ASM,American Samoa,1990,2019,30
1,COK,Cook Islands,1990,2019,30
2,FJI,Fiji,1990,2019,30
3,FSM,Micronesia (Federated States of),1990,2019,30
4,KIR,Kiribati,1990,2019,30
5,MHL,Marshall Islands,1990,2019,30
6,NIU,Niue,1990,2019,30
7,NRU,Nauru,1990,2019,30
8,PLW,Palau,1990,2019,30
9,PYF,French Polynesia,1990,2019,30


## 5. Export cleaned dataset

In [5]:
hypertension_pacific.to_csv(
    "processed_data/adult_hypertension.csv",
    index=False
)

coverage.to_csv(
    "processed_data/adult_hypertension_coverage.csv",
    index=False
)

print("Notebook 3 complete.")
print("Files saved:")
print("- processed_data/adult_hypertension.csv")
print("- processed_data/adult_hypertension_coverage.csv")

Notebook 3 complete.
Files saved:
- processed_data/adult_hypertension.csv
- processed_data/adult_hypertension_coverage.csv


# Notebook Summary

This notebook successfully:

- Downloaded NCD-RisC age-standardized adult hypertension data.
- Filtered to the 16 Pacific Island jurisdictions.
- Averaged male and female estimates to create a single country-year dataset.
- Produced five hypertension indicators:
  - Adult hypertension prevalence
  - Hypertension diagnosis rate
  - Hypertension treatment rate
  - Hypertension control rate
  - Untreated Stage 2 hypertension
- Verified complete coverage from 1990–2019 with no missing values.
- Exported the cleaned dataset for downstream analysis.